# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'resume', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'portfolio project',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio project', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'external affiliation',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [10]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Zhihu page', 'url': 'https://www.zhihu.com/org/huggingface'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'API endpoints page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'Discord join page', 'url': 'https://huggingface.co/join/discord'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
google/diffusiongemma-26B-A4B-it
Updated
5 days ago
•
312k
•
851
MiniMaxAI/MiniMax-M3
Updated
about 9 hours ago
•
14.3k
•
782
moonshotai/Kimi-K2.7-Code
Updated
about 9 hours ago
•
56.8k
•
714
nvidia/LocateAnything-3B
Updated
3 days ago
•
87k
•
2.04k
yuxinlu1/gemma-4-12B-coder-fable5-composer2.

In [13]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ngoogle/diffusiongemma-26B-A4B-it\nUpdated\n5 days ago\n•\n312k\n•\n851\nMiniMaxAI/MiniMax-M3\nUpdated\nabout 9 hours ago\n•\n1

In [16]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Hugging Face – The AI Community Building the Future

Welcome to Hugging Face, the vibrant collaboration platform at the heart of the AI revolution. Hugging Face empowers machine learning engineers, data scientists, researchers, and developers worldwide to create, share, discover, and collaborate on cutting-edge AI models, datasets, and applications. Join a fast-growing global community dedicated to building an open, innovative, and ethical AI future.

---

## About Hugging Face

**Mission:**  
Hugging Face is on a mission to democratize machine learning (ML) by fostering an open platform where anyone—from seasoned ML engineers to enthusiastic learners—can share their work and collaborate effortlessly. Their Hub hosts over 2 million models and half a million datasets, forming the world’s most comprehensive, community-driven machine learning ecosystem.

**Core Offerings:**  
- **Models:** Access 2M+ pre-trained models across diverse AI tasks including NLP, computer vision, audio, and beyond.  
- **Datasets:** Explore and contribute to 500k+ datasets to power your AI projects.  
- **Spaces:** Deploy and interact with community-built AI applications instantly.  
- **Inference Endpoints & Providers:** Enterprise-grade API services to integrate ML models into production environments with ease.  
- **Storage Buckets:** Scalable data storage solutions designed for ML workloads.  
- **HuggingChat:** Cutting-edge conversational AI powered by open models.

**Community & Open Source:**  
Hugging Face thrives on a passionate and welcoming community. Their forum, Discord, and GitHub engagement foster continuous learning and collaboration, while community blog articles and daily research paper summaries keep users informed on AI breakthroughs.

---

## Why Choose Hugging Face?

- **Open & Ethical AI:** The platform promotes transparency and ethical considerations in AI development, encouraging sustainable, responsible use of technology.  
- **Collaboration at Scale:** Whether you’re sharing models or datasets, contributing to research, or deploying AI applications, Hugging Face’s infrastructure supports seamless collaboration.  
- **For Individuals & Enterprises:** From learners and hobbyists to large companies, Hugging Face offers tailored tools—including PRO features and enterprise support—to meet diverse needs.  
- **Cutting-Edge Research:** Partnering with top research labs and hosting state-of-the-art open-source libraries, Hugging Face stays at the forefront of AI innovation.

---

## Company Culture

At Hugging Face, culture centers on openness, generosity, and curiosity. The team embraces diversity and believes in collective intelligence—advancing AI is a community effort. Transparency, inclusivity, and ethical integrity guide their approach internally and across their vibrant external community.

---

## Customers & Partnerships

Hugging Face supports a broad spectrum of users:

- **Researchers & Data Scientists:** To prototype and benchmark ML models efficiently.  
- **Developers & Engineers:** Offering ready-to-use APIs, inference endpoints, and deployment tools for production-ready AI.  
- **Enterprises & AI Labs:** With multi-million dollar partnerships, customized enterprise solutions, and robust support to integrate AI at scale.  
- **Educators & Students:** Providing accessible resources and models for learning and experimentation.

Notable partners include AI labs and technology companies that trust Hugging Face for scalable storage, advanced inference, and collaboration tools.

---

## Careers at Hugging Face

Are you passionate about advancing the future of AI? Hugging Face is constantly growing and invites talented researchers, engineers, developer advocates, and community managers to join their dynamic team.

**Why work here?**  
- Work at the cutting edge of AI technology.  
- Contribute openly to community-driven projects.  
- Thrive in an inclusive and innovative environment.  
- Collaborate with leading experts worldwide.

Explore exciting opportunities in research, software engineering, community engagement, and enterprise solutions on their careers page.

---

## Connect with Hugging Face

- **Website:** [huggingface.co](https://huggingface.co)  
- **Community:** Join on Discord, Forum, and GitHub  
- **Blog:** Stay updated with AI insights and case studies  
- **Social Media:** Follow on Twitter and LinkedIn for latest news and events  

---

**Discover. Create. Collaborate.**  
Join Hugging Face today and be part of the AI future—open, ethical, and powered by community.

---

*Hugging Face — Empowering the machine learning community to build smarter, fairer AI together.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the AI community building the future — a collaborative platform where the global machine learning community creates, discovers, and shares models, datasets, and applications. Positioned as the home of machine learning, Hugging Face empowers both researchers and enterprises to innovate faster by providing accessible, open-source tools and resources.

With over 2 million models, 500,000+ datasets, and 1 million+ AI applications available, Hugging Face is the leading hub for AI models, datasets, and ML apps where collaboration and knowledge sharing accelerate the development and deployment of machine learning solutions.

---

## What We Offer

### Collaborative Platform

- **Models:** Browse and contribute to a vast library of community-curated models, including cutting-edge innovations like Google’s diffusion models and NVIDIA’s LocateAnything.
- **Datasets:** Access extensive datasets updated frequently, supporting everything from coding to vision research.
- **Spaces:** Host and explore interactive ML applications built by the community.
- **Buckets:** Secure and scalable storage solutions for datasets and model assets.

### Enterprise Solutions

Hugging Face provides tailored enterprise support and professional services including Hugging Face PRO, inference endpoints, and integration with cloud providers — ensuring ML deployment is easy, reliable, and scalable.

---

## Community & Culture

Hugging Face fosters an inclusive, open collaboration culture. The company thrives on community contributions, forums, and discussions through platforms like Discord, GitHub, and Forums. It positions itself not just as a technology company but as a vibrant AI community hub, bringing researchers, engineers, and enthusiasts together worldwide.

The culture encourages transparency, shared learning, and innovation, supporting growth with extensive documentation, educational resources, blog posts, daily papers, and direct community support.

---

## Our Customers

Hugging Face serves a diverse audience ranging from academic researchers and open-source developers to businesses and enterprises looking to integrate advanced AI capabilities into their products. Leaders in various industries rely on Hugging Face to accelerate their AI-related projects, ensuring access to the latest AI models and infrastructure with ease.

---

## Careers at Hugging Face

Join a forward-thinking, mission-driven team shaping the future of machine learning. Hugging Face is continuously growing and offers opportunities across AI research, engineering, product development, and community engagement roles. Employees enjoy a culture centered around collaboration, innovation, and impact in the fast-evolving AI ecosystem.

Explore career openings and become part of the AI community building tomorrow’s technology today.

---

## Brand Identity

- **Colors:** Signature yellow (#FFD21E), vibrant orange (#FF9D00), and neutral gray (#6B7280).
- **Logo & Assets:** Available in various formats (.svg, .png, .ai) for use in projects, reflecting the company’s friendly and open brand voice.

---

## Connect with Hugging Face

- Visit: [huggingface.co](https://huggingface.co)
- Engage via their vibrant community channels on Discord, GitHub, and forums.
- Follow blogs, daily papers, and participate in community learning and events.

---

### Hugging Face - The AI Community Building the Future

The preferred destination where machine learning professionals and enthusiasts meet, innovate, and collaborate — driving the future of AI, together.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>